# Multi-Mode Interference & Associative Recall

**Phase 1 Investigation**: Can multiple eigenmodes coexist in a cavity and enable
content-addressable memory via interference?

This notebook tests:
1. Write → hold → read fidelity for single patterns
2. Retrieval fidelity under noise and damping
3. Associative recall: partial query → correct pattern match
4. Capacity limits: how many patterns before fidelity collapses

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from simulations.interference import (
    encode_data, evolve_superposition, readout_modes,
    write_read_verify, associative_recall, fidelity_sweep,
    generate_mode_spectrum,
)

np.random.seed(42)
print('Multi-mode interference module loaded.')

## 1. Single Pattern Write-Read

Encode a 10-element data vector as mode amplitudes, evolve, and read back.

In [ ]:
data = np.array([0.5, 0.8, 0.3, 1.0, 0.6, 0.9, 0.2, 0.7, 0.4, 0.1])

result = write_read_verify(data, Q=500, t_hold=1e-4, noise=0.0)
enc = result['encoding']
res = result['result']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time-domain signal
axes[0, 0].plot(res.time * 1000, res.signal, 'b-', lw=0.5)
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('Superposed Signal u(t)')

# Individual modes
for i in range(min(5, enc.n_modes)):
    axes[0, 1].plot(res.time * 1000, res.mode_signals[i], lw=0.5, label=f'Mode {i+1}')
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].set_title('Individual Mode Contributions')
axes[0, 1].legend(fontsize=8)

# Original vs retrieved amplitudes
x = np.arange(enc.n_modes)
axes[1, 0].bar(x - 0.2, enc.amplitudes, 0.35, label='Original', color='steelblue')
axes[1, 0].bar(x + 0.2, res.retrieved_amplitudes, 0.35, label='Retrieved', color='coral')
axes[1, 0].set_xlabel('Mode Index')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].set_title('Amplitude Recovery')
axes[1, 0].legend()

# FFT showing mode peaks
freqs_fft = np.fft.rfftfreq(len(res.time), res.time[1] - res.time[0])
spectrum = np.abs(np.fft.rfft(res.signal))
axes[1, 1].plot(freqs_fft / 1000, spectrum, 'b-', lw=0.5)
for i, f in enumerate(enc.frequencies):
    axes[1, 1].axvline(x=f/1000, color='r', ls=':', alpha=0.3)
axes[1, 1].set_xlabel('Frequency (kHz)')
axes[1, 1].set_ylabel('|FFT|')
axes[1, 1].set_title('Frequency Spectrum')
axes[1, 1].set_xlim(0, enc.frequencies[-1] * 1.2 / 1000)

plt.suptitle(f'Write-Read Fidelity: {result["fidelity"]:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Fidelity: {result["fidelity"]:.4f}')
print(f'SNR: {result["snr_db"]:.1f} dB')

## 2. Fidelity Under Stress

How does retrieval fidelity degrade with Q, hold time, and noise?

In [ ]:
data = np.random.rand(10)
sweeps = fidelity_sweep(data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Q sweep
Q_vals, fid_Q = sweeps['Q']
axes[0].semilogx(Q_vals, fid_Q, 'g-o', markersize=4)
axes[0].axhline(y=0.8, color='r', ls='--', label='80% threshold')
axes[0].axvline(x=500, color='gray', ls=':', label='Paper Q=500')
axes[0].set_xlabel('Quality Factor Q')
axes[0].set_ylabel('Retrieval Fidelity')
axes[0].set_title('Fidelity vs Q')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hold time sweep
t_vals, fid_t = sweeps['hold_time']
axes[1].semilogx(t_vals * 1000, fid_t, 'b-o', markersize=4)
axes[1].axhline(y=0.8, color='r', ls='--', label='80% threshold')
axes[1].set_xlabel('Hold Time (ms)')
axes[1].set_ylabel('Retrieval Fidelity')
axes[1].set_title('Fidelity vs Hold Time')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Noise sweep
n_vals, fid_n = sweeps['noise']
axes[2].semilogx(n_vals, fid_n, 'r-o', markersize=4)
axes[2].axhline(y=0.8, color='gray', ls='--', label='80% threshold')
axes[2].set_xlabel('Noise Amplitude')
axes[2].set_ylabel('Retrieval Fidelity')
axes[2].set_title('Fidelity vs Noise')
axes[2].set_ylim(0, 1.05)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find thresholds
idx_Q = np.searchsorted(-fid_Q, -0.8)  # first index where fidelity >= 0.8 from left
idx_t = len(fid_t) - 1 - np.searchsorted(fid_t[::-1], 0.8)  # last index >= 0.8
print(f'Q must be > ~{Q_vals[max(0, idx_Q-1)]:.0f} for >80% fidelity')
if idx_t < len(t_vals):
    print(f'Hold time can be up to ~{t_vals[idx_t]*1000:.2f} ms before fidelity drops below 80%')

## 3. Associative Recall

Store 5 patterns, then query with a partial/noisy version. Does interference
correctly identify the closest stored pattern?

In [ ]:
n_modes = 8
n_patterns = 5

# Generate orthogonal-ish random patterns
patterns_data = [np.random.rand(n_modes) for _ in range(n_patterns)]
stored = [encode_data(d, n_modes=n_modes, Q=500) for d in patterns_data]

# Test: query with noisy version of pattern #2
target_idx = 2
query = patterns_data[target_idx] + np.random.normal(0, 0.1, n_modes)
query = np.abs(query)  # keep positive

best_idx, best_overlap, overlaps = associative_recall(stored, query, n_modes=n_modes, Q=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlap scores
colors = ['coral' if i == target_idx else 'steelblue' for i in range(n_patterns)]
colors[best_idx] = 'green'
axes[0].bar(range(n_patterns), overlaps, color=colors)
axes[0].set_xlabel('Pattern Index')
axes[0].set_ylabel('Overlap Score')
axes[0].set_title(f'Associative Recall (target=#{target_idx}, matched=#{best_idx})')
axes[0].set_ylim(0, 1.1)

# Query vs matched pattern
x = np.arange(n_modes)
axes[1].bar(x - 0.2, patterns_data[target_idx], 0.25, label='Original', color='coral')
axes[1].bar(x, np.abs(query), 0.25, label='Query (noisy)', color='gold')
axes[1].bar(x + 0.2, patterns_data[best_idx], 0.25, label='Retrieved', color='green')
axes[1].set_xlabel('Mode Index')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Pattern Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

correct = best_idx == target_idx
print(f'Target pattern: #{target_idx}, Best match: #{best_idx} — {"✅ CORRECT" if correct else "❌ WRONG"}')
print(f'Overlap: {best_overlap:.4f}')

## 4. Capacity Test

How many patterns can we store before recall accuracy drops below 80%?

In [ ]:
n_modes = 10
n_trials = 50
pattern_counts = [2, 3, 5, 8, 10, 15, 20, 30, 50]

accuracies = []
for n_pat in pattern_counts:
    correct = 0
    for trial in range(n_trials):
        patterns_data = [np.random.rand(n_modes) for _ in range(n_pat)]
        stored = [encode_data(d, n_modes=n_modes, Q=500) for d in patterns_data]

        target = np.random.randint(0, n_pat)
        query = patterns_data[target] + np.random.normal(0, 0.15, n_modes)
        query = np.abs(query)

        best_idx, _, _ = associative_recall(stored, query, n_modes=n_modes, Q=500)
        if best_idx == target:
            correct += 1
    accuracies.append(correct / n_trials)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pattern_counts, accuracies, 'bo-', markersize=6, lw=2)
ax.axhline(y=0.8, color='r', ls='--', label='80% accuracy threshold')
ax.set_xlabel('Number of Stored Patterns')
ax.set_ylabel('Recall Accuracy')
ax.set_title(f'Associative Recall Capacity ({n_modes} modes)')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find capacity
for i, (n, acc) in enumerate(zip(pattern_counts, accuracies)):
    print(f'  {n:>3} patterns: {acc*100:.0f}% accuracy')
    if acc < 0.8 and i > 0 and accuracies[i-1] >= 0.8:
        print(f'  → Capacity limit: ~{pattern_counts[i-1]} patterns at 80% accuracy')